In [ ]:
# This file is ran in google colab, since tensorflow would need an older python
# environment to be able to use a gpu on windows
# Also we work on different os, one solution that works on windows might not work on mac
# and thus we reached the conclusion that we should use either google colab or kaggle

# Drive folder location: https://drive.google.com/drive/folders/1KssUFsJxTkS7qAYOACIzuHAcQmMzO8xU?usp=drive_link

import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Conv2D,MaxPooling2D, Flatten, BatchNormalization, Dropout

In [ ]:
# Mount to the drive of the one who runs the code, must conatain "IML/Training.zip & Validation.zip" to function
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Unzip training data
!unzip -q "/content/drive/MyDrive/IML/Training.zip" -d "/content/dataset/"

# Unzip validation data
!unzip -q "/content/drive/MyDrive/IML/Validation.zip" -d "/content/dataset/"

In [ ]:

# create batches of size 32 and resize to (256, 256)
train_dataset= keras.utils.image_dataset_from_directory(
    directory= '/content/dataset/Training',
    labels= 'inferred',
    label_mode = 'int',
    batch_size= 32,
    image_size = (256,256)
)


valid_dataset= keras.utils.image_dataset_from_directory(
    directory= '/content/dataset/Validation',
    labels= 'inferred',
    label_mode = 'int',
    batch_size= 32,
    image_size = (256,256)
)

Found 47009 files belonging to 2 classes.
Found 11649 files belonging to 2 classes.


In [ ]:
# Normalisation of data, cast to float32 (instead of 64?)

def process(image,label):
    image=tf.cast(image/255. , tf.float32)
    return image, label

train_dataset=train_dataset.map(process)
valid_dataset=valid_dataset.map(process)

In [14]:

# Increasing number of filters of sequential Conv2D for more and more complex features
model = Sequential()

# 1st 2D convolution part
model.add(Conv2D(32, kernel_size=(2,2),padding='valid',activation='relu', input_shape=(256,256,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='valid'))

# 2nd 2D convolution part
model.add(Conv2D(64, kernel_size=(2,2),padding='valid',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='valid'))

# 3rd 2D convolution part
model.add(Conv2D(128, kernel_size=(2,2),padding='valid',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='valid'))

# create a 1D vector usable by the Dense layer
model.add(Flatten())

# Gradually decrease the dim and focus on more important features this way
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(1, activation='sigmoid'))

# Show the model layers
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 255, 255, 32)   │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 255, 255, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 126, 126, 64)   │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 126, 126, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 63, 63, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 62, 62, 128)    │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 62, 62, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 31, 31, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 123008)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │    15,745,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,795,937 (60.26 MB)

 Trainable params: 15,795,489 (60.26 MB)

 Non-trainable params: 448 (1.75 KB)

In [15]:
# Compile
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [16]:
# train
history= model.fit(train_dataset, epochs=10, validation_data=valid_dataset)

Epoch 1/10
1470/1470 ━━━━━━━━━━━━━━━━━━━━ 140s 88ms/step - accuracy: 0.8744 - loss: 0.6016 - val_accuracy: 0.9326 - val_loss: 0.1779
Epoch 2/10
1470/1470 ━━━━━━━━━━━━━━━━━━━━ 121s 82ms/step - accuracy: 0.9266 - loss: 0.1962 - val_accuracy: 0.9379 - val_loss: 0.1626
Epoch 3/10
1470/1470 ━━━━━━━━━━━━━━━━━━━━ 115s 78ms/step - accuracy: 0.9371 - loss: 0.1719 - val_accuracy: 0.7694 - val_loss: 0.5662
Epoch 4/10
1470/1470 ━━━━━━━━━━━━━━━━━━━━ 116s 79ms/step - accuracy: 0.9452 - loss: 0.1475 - val_accuracy: 0.8019 - val_loss: 0.4123
Epoch 5/10
1470/1470 ━━━━━━━━━━━━━━━━━━━━ 116s 79ms/step - accuracy: 0.9528 - loss: 0.1297 - val_accuracy: 0.9314 - val_loss: 0.1838
Epoch 6/10
1470/1470 ━━━━━━━━━━━━━━━━━━━━ 115s 79ms/step - accuracy: 0.9589 - loss: 0.1096 - val_accuracy: 0.9462 - val_loss: 0.1484
Epoch 7/10
1470/1470 ━━━━━━━━━━━━━━━━━━━━ 115s 78ms/step - accuracy: 0.9656 - loss: 0.0951 - val_accuracy: 0.9428 - val_loss: 0.1801
Epoch 8/10
1470/1470 ━━━━━━━━━━━━━━━━━━━━ 115s 78ms/step - accuracy: 

In [17]:
# Save the model
model.save('/content/drive/MyDrive/IML/gender_classification_epoch_10.keras')